# ECG Preprocessing Pipeline

In [ ]:
"""
Notebook: 02_preprocessing_pipeline.ipynb
ECG Signal Preprocessing Pipeline
"""

# This notebook demonstrates the complete preprocessing pipeline:
 - Filtering (bandpass, notch)
 - Baseline wander removal
 - R-peak detection
 - Beat segmentation

# 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import sys

sys.path.insert(0, '..')

from src.data.preprocessor import ECGPreprocessor, PreprocessingConfig
from src.data.segmenter import ECGSegmenter
from src.data.loader import ECGLoader

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# 2. Load Raw ECG Signal

In [ ]:
# Load a sample record
loader = ECGLoader(data_dir="../data/raw")
record = loader.load_mit_bih_record(100, leads=[0])
raw_signal = record.signal[:int(10 * record.sampling_rate), 0]

print(f"Raw signal shape: {raw_signal.shape}")
print(f"Sampling rate: {record.sampling_rate} Hz")

# 3. Visualize Raw Signal

In [ ]:
time = np.arange(len(raw_signal)) / record.sampling_rate

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(time, raw_signal, 'b-', linewidth=1)
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('Amplitude (mV)')
ax.set_title('Raw ECG Signal (with noise and baseline wander)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 4. Apply Preprocessing Steps

In [ ]:
# Initialize preprocessor
preprocessor = ECGPreprocessor(sampling_rate=record.sampling_rate)

# Apply each step and visualize
fig, axes = plt.subplots(5, 1, figsize=(14, 12))

# Step 1: Raw signal
axes[0].plot(time[:1000], raw_signal[:1000], 'b-', linewidth=1)
axes[0].set_ylabel('Raw (mV)')
axes[0].set_title('Step 1: Raw ECG Signal')
axes[0].grid(True, alpha=0.3)

# Step 2: Baseline removal
baseline_removed = preprocessor.remove_baseline_wander(raw_signal)
axes[1].plot(time[:1000], baseline_removed[:1000], 'g-', linewidth=1)
axes[1].set_ylabel('Baseline Removed (mV)')
axes[1].set_title('Step 2: Baseline Wander Removed')
axes[1].grid(True, alpha=0.3)

# Step 3: Notch filter
notch_filtered = preprocessor.apply_notch_filter(baseline_removed)
axes[2].plot(time[:1000], notch_filtered[:1000], 'orange', linewidth=1)
axes[2].set_ylabel('Notch Filtered (mV)')
axes[2].set_title('Step 3: Powerline Noise Removed (50/60 Hz)')
axes[2].grid(True, alpha=0.3)

# Step 4: Bandpass filter
bandpass_filtered = preprocessor.apply_bandpass_filter(notch_filtered)
axes[3].plot(time[:1000], bandpass_filtered[:1000], 'r-', linewidth=1)
axes[3].set_ylabel('Bandpass (mV)')
axes[3].set_title('Step 4: Bandpass Filter (0.5-40 Hz)')
axes[3].grid(True, alpha=0.3)

# Step 5: Normalized
normalized = preprocessor.normalize_signal(bandpass_filtered)
axes[4].plot(time[:1000], normalized[:1000], 'purple', linewidth=1)
axes[4].set_xlabel('Time (seconds)')
axes[4].set_ylabel('Normalized')
axes[4].set_title('Step 5: Z-Score Normalization')
axes[4].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 5. Frequency Domain Analysis

In [ ]:
# Compare frequency spectra before and after filtering
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Compute FFT
freqs = np.fft.rfftfreq(len(raw_signal), 1/record.sampling_rate)
raw_fft = np.abs(np.fft.rfft(raw_signal))
processed_fft = np.abs(np.fft.rfft(normalized))

axes[0].plot(freqs, 20 * np.log10(raw_fft + 1e-10), 'b-', linewidth=1)
axes[0].set_xlim(0, 100)
axes[0].set_ylabel('Magnitude (dB)')
axes[0].set_title('Frequency Spectrum - Raw Signal')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(50, color='red', linestyle='--', alpha=0.5, label='50 Hz noise')
axes[0].legend()

axes[1].plot(freqs, 20 * np.log10(processed_fft + 1e-10), 'r-', linewidth=1)
axes[1].set_xlim(0, 100)
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Magnitude (dB)')
axes[1].set_title('Frequency Spectrum - Processed Signal')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 6. R-Peak Detection

In [ ]:
# Initialize segmenter
segmenter = ECGSegmenter(sampling_rate=record.sampling_rate)

# Detect R-peaks
r_peaks, peak_info = segmenter.detect_r_peaks(normalized, method='neurokit2')

print(f"Detected {len(r_peaks)} R-peaks in 10-second segment")
print(f"Heart rate: {peak_info.get('heart_rate_bpm', 0):.1f} BPM")

# Visualize detected peaks
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(time, normalized, 'b-', linewidth=1, alpha=0.7, label='ECG')
ax.scatter(time[r_peaks], normalized[r_peaks], 
          c='red', s=50, marker='^', zorder=5, label='Detected R-peaks')
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('Amplitude (mV)')
ax.set_title(f'R-Peak Detection - {len(r_peaks)} beats detected')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 7. Beat Segmentation

In [ ]:
# Segment individual beats
beats = segmenter.segment_beats(normalized, r_peaks)
print(f"Segmented {len(beats)} beats")

# Visualize first 5 beats
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
axes = axes.flatten()

for i, beat in enumerate(beats[:5]):
    beat_time = np.arange(len(beat.signal)) / record.sampling_rate * 1000  # ms
    axes[i].plot(beat_time, beat.signal, 'b-', linewidth=1.5)
    axes[i].axvline(beat_time[beat.r_peak_index], color='red', 
                   linestyle='--', alpha=0.7, label='R-peak')
    axes[i].set_xlabel('Time (ms)')
    axes[i].set_ylabel('Amplitude (mV)')
    axes[i].set_title(f'Beat {i+1} (Quality: {beat.quality_score:.2f})')
    axes[i].legend(fontsize=8)
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 8. Beat Quality Assessment

In [ ]:
# Analyze beat quality scores
quality_scores = [b.quality_score for b in beats]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(quality_scores, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
ax1.set_xlabel('Quality Score')
ax1.set_ylabel('Frequency')
ax1.set_title('Beat Quality Distribution')
ax1.axvline(np.median(quality_scores), color='red', linestyle='--', 
           label=f'Median: {np.median(quality_scores):.3f}')
ax1.legend()

ax2.boxplot(quality_scores, vert=True)
ax2.set_ylabel('Quality Score')
ax2.set_title('Beat Quality Box Plot')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"Mean quality score: {np.mean(quality_scores):.3f}")
print(f"Median quality score: {np.median(quality_scores):.3f}")
print(f"Low quality beats (<0.5): {sum(1 for q in quality_scores if q < 0.5)}")

# 9. Save Processed Data

In [ ]:
# Save processed beats for training
output_dir = Path("../data/processed/demo")
output_dir.mkdir(parents=True, exist_ok=True)

beat_signals = np.array([b.signal for b in beats])
beat_labels = np.array([0] * len(beats))  # Placeholder labels

np.save(output_dir / "demo_beats.npy", beat_signals)
np.save(output_dir / "demo_labels.npy", beat_labels)

print(f"Saved {len(beats)} beats to {output_dir}")